In [ ]:
# imports
import sys
import numpy as np
import pandas as pd
import scipy.linalg as slin

sys.path.append('helpers/pcca_fa/')
from dual_pfc_funcs import getParams, gen_GP, load_dict, getParams
import helpers.pcca_fa.pcca_fa_mdl as pf
import helpers.pcca_fa.cca.prob_cca as pcca

import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
plt.style.use('scifigs.mplstyle')
SAVE_FIG = False

# params
color_map = getParams()['color_map']
num_neurons = 20
num_timepoints = 1000
GP_tau = 50
base_p = 1e-2
c = 5
base_add = 0.05

In [ ]:
# create figure to plot generated latents
fig,ax = plt.subplots(3,1,tight_layout=True)

# generate across-area latent
across_z = gen_GP(GP_tau, num_timepoints, seed=0)
ax[0].plot(across_z,color=color_map['across'])

# generate within-area 1 latent
within_z = gen_GP(GP_tau, num_timepoints, seed=2, N=2)
within_z1 = within_z[:,0]
ax[1].plot(within_z1,color=color_map['within1'])

# generate within-area 2 latent
within_z2 = within_z[:,1]
ax[2].plot(within_z2,color=color_map['within2'])

if SAVE_FIG:
    pdf = PdfPages('figs/sim_latents.pdf')
    pdf.savefig(fig)
    pdf.close()
else:
    fig.show()

In [ ]:
# generate across-area loadings
np.random.seed(0)
across_loadings1 = np.random.randn(num_neurons) + 1
across_loadings1 /= slin.norm(across_loadings1)

across_loadings2 = np.random.randn(num_neurons) + 1
across_loadings2 /= slin.norm(across_loadings2)

# generate within-area 1 loadings
np.random.seed(1)
within_loadings1 = np.random.randn(num_neurons) + 1
within_loadings1 /= slin.norm(within_loadings1)

# generate within-area 2 loadings
np.random.seed(6)
within_loadings2 = np.random.randn(num_neurons) + 1
within_loadings2 /= slin.norm(within_loadings2)

# generate spike trains
area1_raster = np.zeros((num_neurons,num_timepoints))
area2_raster = np.zeros((num_neurons,num_timepoints))

for ineuron in range(num_neurons):
    # area 1
    zi = across_loadings1[ineuron]*across_z + within_loadings1[ineuron]*within_z1
    if np.max(zi) <= 0:
        zi[zi <= 0] = base_p
    else:
        zi = zi / np.max(zi) / c # control overall firing by d
        zi = zi + base_add
    zi[zi <= 0] = base_p
    area1_raster[ineuron,:] = np.random.binomial(np.ones((num_timepoints,)).astype('int'), zi)
    
    # area 2
    zi = across_loadings2[ineuron]*across_z + within_loadings2[ineuron]*within_z2
    if np.max(zi) <= 0:
        zi[zi <= 0] = base_p
    else:
        zi = zi / np.max(zi) / c # control overall firing by d
        zi = zi + base_add
    zi[zi <= 0] = base_p
    area2_raster[ineuron,:] = np.random.binomial(np.ones((num_timepoints,)).astype('int'), zi)

In [ ]:
# plot rasters
fig,ax = plt.subplots(2,1,figsize=(4,6),sharex=True,sharey=True)

ax[0].scatter(np.where(area1_raster)[1],np.where(area1_raster)[0],marker='|',color='k',linewidth=0.3)
ax[1].scatter(np.where(area2_raster)[1],np.where(area2_raster)[0],marker='|',color='k',linewidth=0.3)

ax[0].set_title('area 1 neurons',color=color_map['within1'])
ax[0].set_xlim([0,1000])
ax[0].set_ylim([-0.5,num_neurons])
ax[0].set_xticks([])
ax[0].set_yticks([])
ax[0].spines[['left','bottom']].set_visible(False)

ax[1].set_title('area 2 neurons',color=color_map['within2'])
ax[1].set_xticks([])
ax[1].set_yticks([])
ax[1].spines[['left','bottom']].set_visible(False)

if SAVE_FIG:
    pdf = PdfPages('figs/sim_rasters.pdf')
    pdf.savefig(fig)
    pdf.close()
else:
    fig.show()

In [ ]:
# proportion of variance explained
if len(across_loadings1.shape) == 1:
    across_loadings1 = across_loadings1[:,np.newaxis]
    across_loadings2 = across_loadings2[:,np.newaxis]
    within_loadings1 = within_loadings1[:,np.newaxis]
    within_loadings2 = within_loadings2[:,np.newaxis]

across_x1 = np.diag(across_loadings1.dot(across_loadings1.T))
across_x2 = np.diag(across_loadings2.dot(across_loadings2.T))
within_x1 = np.diag(within_loadings1.dot(within_loadings1.T))
within_x2 = np.diag(within_loadings2.dot(within_loadings2.T))

total_x1 = across_x1 + within_x1 + base_add # WWT + LLT + Psi
total_x2 = across_x2 + within_x2 + base_add # WWT + LLT + Psi

n1 = len(total_x1) # number of neurons in area 1 - LEFT
n2 = len(total_x2) # number of neurons in area 2 - RIGHT

d = 1
d1 = 1
d2 = 1

# plot
fig,ax = plt.subplots(2,1,sharex=True)
fig.set_figheight(2*fig.get_figheight())

ax[0].barh(np.arange(1,n1+1,1),total_x1,color=color_map['independent'],label='independent')
ax[0].barh(np.arange(1,n1+1,1),within_x1+across_x1,color=color_map['within1'],label='within')
ax[0].barh(np.arange(1,n1+1,1),across_x1,color=color_map['across'],label='across')
ax[1].barh(np.arange(1,n2+1,1),total_x2,color=color_map['independent'],label='independent')
ax[1].barh(np.arange(1,n2+1,1),within_x2+across_x2,color=color_map['within2'],label='within')
ax[1].barh(np.arange(1,n2+1,1),across_x2,color=color_map['across'],label='across')

# plot formatting
ticks = np.arange(0,max([n1,n2]),10)
ticks[0] = 1
ax[0].invert_yaxis(),ax[1].invert_yaxis()
ax[0].set_yticks(ticks),ax[1].set_yticks(ticks)
ax[0].legend(loc='lower right',frameon=False),ax[1].legend(loc='lower right',frameon=False)
ax[1].set_xticks([])
ax[1].set_ylabel('area 2 neurons',color=color_map['within2'])
ax[0].set_ylabel('area 1 neurons',color=color_map['within1'])
ax[1].spines['bottom'].set_visible(False),ax[1].spines['bottom'].set_visible(False)

if SAVE_FIG:
    pdf = PdfPages('figs/sim_partitions.pdf')
    pdf.savefig(fig)
    pdf.close()
else:
    fig.show()

## Model validation

In [ ]:
# panel e: vary N
data_path = 'preprocessed_data/'
filename = data_path + 'simdataset_varyN_noWS.pkl'
dat = load_dict(filename)
n_boots = dat['n_boots']
n_trials = dat['N']

df = pd.DataFrame(columns=['SimId','NConfig','GT-SvW1','GT-SvW2','Est-SvW1','Est-SvW2','Error-SvW1','Error-SvW2',
                           'GT-SvL1','GT-SvL2','Est-SvL1','Est-SvL2','Error-SvL1','Error-SvL2',
                           'pCCA-SvW1', 'pCCA-SvW2','pCCA-Error-SvW1','pCCA-Error-SvW2',
                           'GT-DSharedW1','GT-DSharedW2','Est-DSharedW1','Est-DSharedW2','Error-DSharedW1','Error-DSharedW2',
                           'GT-DSharedL1','GT-DSharedL2','Est-DSharedL1','Est-DSharedL2','Error-DSharedL1','Error-DSharedL2',
                           'pCCA-DSharedW1', 'pCCA-DSharedW2','pCCA-Error-DSharedW1','pCCA-Error-DSharedW2'])
for idx in range(len(dat['sim_params'])):
    gt_param = dat['sim_params'][idx]
    est_param = dat['est_params'][idx]
    pcca_param = dat['pcca_params'][idx]
    
    # get ground truth %sv
    mdl = pf.pcca_fa()
    mdl.set_params(gt_param)
    psv_gt = mdl.compute_psv()
    dim_gt = mdl.compute_dshared()

    # get est %sv for pCCA-FA
    mdl = pf.pcca_fa()
    mdl.set_params(est_param)
    psv_est = mdl.compute_psv()
    dim_est = mdl.compute_dshared()

    # get est %sv for pCCA :o
    mdl = pcca.prob_cca()
    mdl.set_params(pcca_param)
    psv_pcca = mdl.compute_psv()
    dim_pcca = mdl.compute_dshared()

    df2 = {'SimId':idx,'NConfig':n_trials[idx],
           'GT-SvW1':psv_gt['avg_psv_W_1'],'GT-SvW2':psv_gt['avg_psv_W_2'],'Est-SvW1':psv_est['avg_psv_W_1'],'Est-SvW2':psv_est['avg_psv_W_2'],
           'GT-SvL1':psv_gt['avg_psv_L_1'],'GT-SvL2':psv_gt['avg_psv_L_2'],'Est-SvL1':psv_est['avg_psv_L_1'],'Est-SvL2':psv_est['avg_psv_L_2'],
            'Error-SvW1':psv_est['avg_psv_W_1']-psv_gt['avg_psv_W_1'],'Error-SvW2':psv_est['avg_psv_W_2']-psv_gt['avg_psv_W_2'],
            'Error-SvL1':psv_est['avg_psv_L_1']-psv_gt['avg_psv_L_1'],'Error-SvL2':psv_est['avg_psv_L_2']-psv_gt['avg_psv_L_2'],
            'pCCA-SvW1':psv_pcca['psv_x'], 'pCCA-SvW2':psv_pcca['psv_y'], 'pCCA-Error-SvW1':psv_pcca['psv_x']-psv_gt['avg_psv_W_1'], 'pCCA-Error-SvW2':psv_pcca['psv_y']-psv_gt['avg_psv_W_2'],
            'GT-DSharedW1':dim_gt['dshared_W_1'],'GT-DSharedW2':dim_gt['dshared_W_2'],'Est-DSharedW1':dim_est['dshared_W_1'],'Est-DSharedW2':dim_est['dshared_W_2'],
            'GT-DSharedL1':dim_gt['dshared_L_1'],'GT-DSharedL2':dim_gt['dshared_L_2'],'Est-DSharedL1':dim_est['dshared_L_1'],'Est-DSharedL2':dim_est['dshared_L_2'],
            'Error-DSharedW1':dim_est['dshared_W_1']-dim_gt['dshared_W_1'],'Error-DSharedW2':dim_est['dshared_W_2']-dim_gt['dshared_W_2'],
            'Error-DSharedL1':dim_est['dshared_L_1']-dim_gt['dshared_L_1'],'Error-DSharedL2':dim_est['dshared_L_2']-dim_gt['dshared_L_2'],
            'pCCA-DSharedW1':dim_pcca['dshared_x'], 'pCCA-DSharedW2':dim_pcca['dshared_y'], 'pCCA-Error-DSharedW1':dim_pcca['dshared_x']-dim_gt['dshared_W_1'], 'pCCA-Error-DSharedW2':dim_pcca['dshared_y']-dim_gt['dshared_W_2']}
    df.loc[len(df)] = df2

sv_global_mean, sv_local_mean, pcca_mean = [], [], []
sv_global_std, sv_local_std, pcca_std = [], [], []
dim_global_mean, dim_local_mean, dim_pcca_mean = [], [], []
dim_global_std, dim_local_std, dim_pcca_std = [], [], []
for n in np.unique(n_trials):
    filt = df['NConfig'] == n

    # calculate error in estimated shared variance
    curr_global = np.concatenate((df[filt]['Error-SvW1'],df[filt]['Error-SvW2']))
    curr_local = np.concatenate((df[filt]['Error-SvL1'],df[filt]['Error-SvL2']))
    curr_pcca = np.concatenate((df[filt]['pCCA-Error-SvW1'],df[filt]['pCCA-Error-SvW2']))

    sv_global_mean.append(np.mean(curr_global))
    sv_local_mean.append(np.mean(curr_local))
    pcca_mean.append(np.mean(curr_pcca))
    sv_global_std.append(np.std(curr_global))
    sv_local_std.append(np.std(curr_local))
    pcca_std.append(np.std(curr_pcca))

    # calculate error in estimated dimensionality
    curr_global = np.concatenate((df[filt]['Error-DSharedW1'],df[filt]['Error-DSharedW2']))
    curr_local = np.concatenate((df[filt]['Error-DSharedL1'],df[filt]['Error-DSharedL2']))
    curr_pcca = np.concatenate((df[filt]['pCCA-Error-DSharedW1'],df[filt]['pCCA-Error-DSharedW2']))

    dim_global_mean.append(np.mean(curr_global))
    dim_local_mean.append(np.mean(curr_local))
    dim_pcca_mean.append(np.mean(curr_pcca))
    dim_global_std.append(np.std(curr_global))
    dim_local_std.append(np.std(curr_local))
    dim_pcca_std.append(np.std(curr_pcca))

In [ ]:
fig,ax = plt.subplots(2,1,sharex=True,sharey=True,figsize=(5,6))
fig.tight_layout(pad=2.5)

fig.supylabel('estimation error in %sv')
fig.supxlabel('number of trials used to estimate')

ax[0].plot([0,np.max(n_trials)],[0,0],'--', color='gray',label='ground truth') # across
ax[1].plot([0,np.max(n_trials)],[0,0],'--', color='gray',label='ground truth') # within

xdata = np.unique(n_trials)
ax[0].errorbar(xdata, sv_global_mean, yerr=sv_global_std, fmt='-o', color=color_map['across'], ms=4, label='pCCA-FA')
ax[0].errorbar(xdata, pcca_mean, yerr=pcca_std, fmt='-o', color=color_map['across'], alpha=0.5, ms=4,label='pCCA')
ax[1].errorbar(xdata, sv_local_mean, yerr=sv_local_std, fmt='-o', color=color_map['within'], ms=4, label='pCCA-FA')

ax[0].set_xticks(np.arange(0,np.max(n_trials)+1,500))
ax[0].legend(loc='lower right')
ax[1].legend(loc='lower right')

if SAVE_FIG:
    pdf = PdfPages('figs/sv_error_varyN_noWS.pdf')
    pdf.savefig(fig)
    pdf.close()
else:
    fig.show()

In [ ]:
fig,ax = plt.subplots(2,1,sharex=True,sharey=True,figsize=(5,6))
fig.tight_layout(pad=2.5)

fig.supylabel('error in $d_{shared}$')
fig.supxlabel('number of trials used to estimate')

ax[0].plot([0,np.max(n_trials)],[0,0],'--', color='gray') # across
ax[1].plot([0,np.max(n_trials)],[0,0],'--', color='gray') # within

xdata = np.unique(n_trials)
ax[0].errorbar(xdata, dim_global_mean, yerr=dim_global_std, fmt='-o', color=color_map['across'], ms=4)
ax[0].errorbar(xdata, dim_pcca_mean, yerr=dim_pcca_std, fmt='-o', color=color_map['across'], alpha=0.5, ms=4)
ax[1].errorbar(xdata, dim_local_mean, yerr=dim_local_std, fmt='-o', color=color_map['within'], ms=4)

ax[0].set_xticks(np.arange(0,np.max(n_trials)+1,500))

if SAVE_FIG:
    pdf = PdfPages('figs/dim_error_varyN_noWS.pdf')
    pdf.savefig(fig)
    pdf.close()
else:
    fig.show()